In [0]:
# ============================================================================
# Rolling-window DELTA embeddings -- generalizes the month-to-month delta
# logic to overlapping (rolling) windows, where the correct "previous
# period" isn't 1 step back but N_BACK steps back:
#   - rolling trimester (3-month window, 1-month stride): N_BACK=3, since
#     window[i-3]'s end == window[i]'s start EXACTLY (both built from the
#     same calendar-month DateOffset arithmetic on the same anchor) --
#     subtracting captures the true non-overlapping change with no gap
#     or overlap.
#   - rolling month (1-month window, 1-week stride): N_BACK=4, since
#     4 weeks (28 days, fixed) approximates but does NOT exactly equal
#     1 calendar month (28-31 days) -- see conversation note. Still the
#     best available approximation of "about a month ago" under this
#     stride.
#
# Within the first (N_BACK - 1) available windows of a patient's
# trajectory (not enough history yet for a full N_BACK-step lookback):
#   - the FIRST available window is left as raw cumulative (nothing
#     earlier to subtract, same convention as the original month-to-
#     month delta script)
#   - available windows 2..N_BACK-1 (position 1 through N_BACK-2 within
#     the available run) fall back to a SINGLE-STEP delta (current minus
#     immediately preceding window) rather than the ideal N_BACK-step
#     delta, since the N_BACK-step predecessor doesn't exist yet
#   - from the N_BACK-th available window onward, the full N_BACK-step
#     delta applies
#
# Same non-contiguity invariant as the original script: unavailable can
# only appear as a leading (pre-history) or trailing (data-cutoff) block,
# never a gap between available windows -- verified in the data, not
# assumed, per batch.
# ============================================================================
import gc
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

EMBED_DIM = 768
BATCH_SIZE = 5000
EMB_COLS = [f"emb_{d}" for d in range(EMBED_DIM)]

# ---- window name grids, regenerated to match the extraction pipeline's
# naming convention exactly (month_wk{offset:+03d}, tri_mo{offset:+03d}) ----
MONTH_PRE_WEEKS, MONTH_POST_WEEKS, MONTH_STRIDE_WEEKS = 26, 48, 1
MONTH_WINDOW_ORDER = [f"month_wk{o:+03d}" for o in range(-MONTH_PRE_WEEKS, MONTH_POST_WEEKS, MONTH_STRIDE_WEEKS)]

TRI_PRE_MONTHS, TRI_POST_MONTHS, TRI_STRIDE_MONTHS = 6, 13, 1
TRI_WINDOW_ORDER = [f"tri_mo{o:+03d}" for o in range(-TRI_PRE_MONTHS, TRI_POST_MONTHS, TRI_STRIDE_MONTHS)]


def process_batch_rolling_delta_no_rampup(batch_df, batch_num, window_order, n_back, embed_dim=EMBED_DIM):
    """Same as process_batch_rolling_delta, EXCEPT: no ramp-up fallback.
    Only two kinds of non-NaN rows exist per patient:
      - the first available window -> raw cumulative embedding (base row)
      - the n_back-th available window onward -> true n_back-step delta,
        spanning the FULL intended window duration (3 months for
        trimester, ~1 month for month)
    Any available window strictly between these (positions 1..n_back-1
    of the available run) is left as NaN -- there is no way to compute a
    delta of the intended duration there yet, and a shorter-duration
    delta would misrepresent the window length, which is exactly what
    you want to avoid.
    """
    n_windows = len(window_order)

    if batch_df.duplicated(subset=["patient_id", "window_name"]).any():
        n_dup = batch_df.duplicated(subset=["patient_id", "window_name"]).sum()
        raise ValueError(f"Batch {batch_num}: {n_dup} duplicate (patient_id, window_name) rows.")

    counts = batch_df.groupby("patient_id").size()
    bad_counts = counts[counts != n_windows]
    if len(bad_counts) > 0:
        raise ValueError(f"Batch {batch_num}: {len(bad_counts)} patients don't have exactly "
                         f"{n_windows} rows. Example: {bad_counts.index[:5].tolist()}.")

    unknown_names = set(batch_df["window_name"].unique()) - set(window_order)
    if unknown_names:
        raise ValueError(f"Batch {batch_num}: unrecognized window_name values: {unknown_names}.")

    batch_df = batch_df.copy()
    batch_df["window_name"] = pd.Categorical(batch_df["window_name"], categories=window_order, ordered=True)
    batch_df = batch_df.sort_values(["patient_id", "window_name"]).reset_index(drop=True)

    n_patients = batch_df["patient_id"].nunique()
    patient_order = batch_df["patient_id"].drop_duplicates().tolist()

    emb_matrix = batch_df[EMB_COLS].values.reshape(n_patients, n_windows, embed_dim)
    unavailable_matrix = batch_df["unavailable"].values.reshape(n_patients, n_windows).astype(int)

    avail_indicator = (unavailable_matrix == 0).astype(int)
    padded = np.pad(avail_indicator, ((0, 0), (1, 1)), constant_values=0)
    run_starts = np.diff(padded, axis=1) == 1
    n_available_runs = run_starts.sum(axis=1)
    violation_mask = n_available_runs > 1
    n_violations = int(violation_mask.sum())
    if n_violations > 0:
        violating_pids = np.array(patient_order)[violation_mask][:5]
        print(f"  WARNING (batch {batch_num}): {n_violations} patients have a non-contiguous "
              f"availability pattern. Example patient_ids: {violating_pids.tolist()}.")

    has_any_available = (unavailable_matrix == 0).any(axis=1)
    first_avail_idx = np.argmax(unavailable_matrix == 0, axis=1)

    delta_matrix = np.full_like(emb_matrix, np.nan, dtype=np.float32)

    # true n_back-step deltas -- unchanged from before
    if n_back < n_windows:
        delta_matrix[:, n_back:, :] = emb_matrix[:, n_back:, :] - emb_matrix[:, :-n_back, :]

    # base row: first available window, raw cumulative
    valid_patients = np.where(has_any_available)[0]
    delta_matrix[valid_patients, first_avail_idx[valid_patients], :] = \
        emb_matrix[valid_patients, first_avail_idx[valid_patients], :]

    # ---- NO ramp-up override here (this is the only change vs before) --
    # positions first_avail_idx+1 .. first_avail_idx+n_back-1 stay NaN,
    # since a full n_back-step delta isn't computable there yet and a
    # shorter one would misrepresent the window duration.

    out_df = batch_df.copy()
    out_df[EMB_COLS] = delta_matrix.reshape(n_patients * n_windows, embed_dim)
    out_df["window_name"] = out_df["window_name"].astype(str)

    n_base_rows = int(has_any_available.sum())
    n_delta_rows = int(np.isfinite(delta_matrix).all(axis=2).sum()) - n_base_rows
    n_unavail_rows = int((unavailable_matrix == 1).sum())
    n_rampup_dropped = int((unavailable_matrix == 0).sum()) - n_base_rows - n_delta_rows
    if n_rampup_dropped > 0:
        print(f"  Batch {batch_num}: {n_rampup_dropped} available windows had no full-duration "
              f"predecessor yet -- left as NaN rather than a shorter-window delta.")
    return out_df, n_patients, n_base_rows, n_delta_rows, n_unavail_rows


def run_rolling_delta_pipeline(source_table, output_table, window_order, n_back,
                                embed_dim=EMBED_DIM, batch_size=BATCH_SIZE):
    """Batched driver -- identical structure/memory-bounding to the
    original month-to-month delta cell, parameterized by window_order
    and n_back."""
    print(f"Loading patient list from {source_table}...")
    all_patient_ids = [row.patient_id for row in
                       spark.table(source_table).select("patient_id").distinct().collect()]
    n_total_patients = len(all_patient_ids)
    n_batches = (n_total_patients + batch_size - 1) // batch_size
    print(f"{n_total_patients} patients total -> {n_batches} batches of up to {batch_size}. "
          f"n_back={n_back}, n_windows={len(window_order)}.")

    total_base, total_delta, total_unavail = 0, 0, 0
    for batch_idx in range(n_batches):
        batch_ids = all_patient_ids[batch_idx * batch_size: (batch_idx + 1) * batch_size]

        batch_spark_df = spark.table(source_table).filter(F.col("patient_id").isin(batch_ids))
        batch_df = batch_spark_df.toPandas()

        out_df, n_patients, n_base, n_delta, n_unavail = process_batch_rolling_delta_no_rampup(
            batch_df, batch_idx + 1, window_order, n_back, embed_dim
        )

        write_mode = "overwrite" if batch_idx == 0 else "append"
        spark.createDataFrame(out_df).write.format("delta").mode(write_mode).saveAsTable(output_table)

        total_base += n_base
        total_delta += n_delta
        total_unavail += n_unavail
        print(f"Batch {batch_idx + 1}/{n_batches}: {n_patients} patients written "
              f"({n_base} base, {n_delta} delta, {n_unavail} unavailable rows) "
              f"-- mode={write_mode}")

        del batch_spark_df, batch_df, out_df
        gc.collect()

    print(f"\nDone. {n_total_patients} patients written to {output_table}.")
    print(f"Totals: {total_base} base (raw, first-available) rows, "
          f"{total_delta} true delta rows, {total_unavail} unavailable (NaN) rows.")

In [0]:
run_rolling_delta_pipeline(
    source_table="catalog.schema.patient_clmbr_embeddings_new_v5_rolling_trimester",
    output_table="catalog.schema.patient_clmbr_embeddings_new_v5_rolling_trimester_delta",
    window_order=TRI_WINDOW_ORDER,
    n_back=3,
)

In [0]:
run_rolling_delta_pipeline(
    source_table="catalog.schema.patient_clmbr_embeddings_new_v5_rolling_month",
    output_table="catalog.schema.patient_clmbr_embeddings_new_v5_rolling_month_delta",
    window_order=MONTH_WINDOW_ORDER,
    n_back=4,
)